In [18]:
import sys
import os

sys.path.append(os.path.abspath('..'))



from helpers.mysql_connections import create_mysql_engine

import pandas as pd
import matplotlib.pyplot as plt

engine = create_mysql_engine()

templates_query = 'SELECT * FROM gophish.templates;'
df_templates = pd.read_sql( templates_query, engine )
df_templates.head(10)


,id,user_id,name,subject,text,html,modified_date,envelope_sender
0,1,1,Template P1-1,Actualizacion de cuenta corporativa P1-1,Actualizacion de cuenta corporativa P1-1. Se s...,<html><body><h2>Actualizacion de cuenta corpor...,2021-02-07 08:00:00,None
1,2,1,Template P1-2,Revision de acceso a VPN P1-2,Revision de acceso a VPN P1-2. Se solicita val...,<html><body><h2>Revision de acceso a VPN P1-2<...,2021-02-14 08:00:00,None
2,3,3,Template P1-3,Confirmacion de nomina interna P1-3,Confirmacion de nomina interna P1-3. Se solici...,<html><body><h2>Confirmacion de nomina interna...,2021-02-21 08:00:00,None
3,4,2,Template P1-4,Verificacion de correo empresarial P1-4,Verificacion de correo empresarial P1-4. Se so...,<html><body><h2>Verificacion de correo empresa...,2021-02-28 08:00:00,None
4,5,2,Template P1-5,Politica nueva de seguridad P1-5,Politica nueva de seguridad P1-5. Se solicita ...,<html><body><h2>Politica nueva de seguridad P1...,2021-03-07 08:00:00,None
5,6,2,Template P1-6,Portal de vacaciones y permisos P1-6,Portal de vacaciones y permisos P1-6. Se solic...,<html><body><h2>Portal de vacaciones y permiso...,2021-03-14 08:00:00,None
6,7,1,Template P1-7,Actualizacion de beneficios internos P1-7,Actualizacion de beneficios internos P1-7. Se ...,<html><body><h2>Actualizacion de beneficios in...,2021-03-21 08:00:00,None
7,8,1,Template P1-8,Revision de firma electronica P1-8,Revision de firma electronica P1-8. Se solicit...,<html><body><h2>Revision de firma electronica ...,2021-03-28 08:00:00,None
8,9,4,Template P1-9,Confirmacion de acceso remoto P1-9,Confirmacion de acceso remoto P1-9. Se solicit...,<html><body><h2>Confirmacion de acceso remoto ...,2021-04-04 08:00:00,None
9,10,1,Template P1-10,Validacion de directorio interno P1-10,Validacion de directorio interno P1-10. Se sol...,<html><body><h2>Validacion de directorio inter...,2021-04-11 08:00:00,None


In [8]:

import re
import unicodedata
import pandas as pd
import numpy as np

# -----------------------------
# 2. Cargar tablas
# -----------------------------
df_templates = pd.read_sql("""
    SELECT id, user_id, name, subject, text, html, modified_date
    FROM templates
""", engine)

df_cls = pd.read_sql("""
    SELECT id, text, context, subcontext, risk, message_type, text_level,
           intention, action_required, is_urgent, has_url, has_credentials
    FROM text_clasification
""", engine)

# -----------------------------
# 3. Funciones de limpieza
# -----------------------------
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()

    # quitar acentos
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )

    # reemplazar saltos y tabs por espacio
    s = re.sub(r'[\r\n\t]+', ' ', s)

    # quitar espacios duplicados
    s = re.sub(r'\s+', ' ', s)

    return s

df_templates["text_norm"] = df_templates["text"].apply(normalize_text)
df_cls["pattern_norm"] = df_cls["text"].apply(normalize_text)

# -----------------------------
# 4. Preparar patrones
# -----------------------------
# Ordenar por longitud descendente ayuda a detectar primero frases largas
df_cls["pattern_len"] = df_cls["pattern_norm"].str.len()
df_cls = df_cls.sort_values("pattern_len", ascending=False).reset_index(drop=True)

# -----------------------------
# 5. Buscar coincidencias
# -----------------------------
matches = []

for _, tpl in df_templates.iterrows():
    template_id = tpl["id"]
    template_name = tpl["name"]
    template_subject = tpl["subject"]
    original_text = tpl["text"]
    text_norm = tpl["text_norm"]

    for _, cls in df_cls.iterrows():
        pattern = cls["pattern_norm"]

        if not pattern:
            continue

        # Si es una sola palabra, usar bordes de palabra
        if " " not in pattern:
            regex = rf"\b{re.escape(pattern)}\b"
        else:
            regex = re.escape(pattern)

        found = list(re.finditer(regex, text_norm))


        if found:
            for m in found:
                matches.append({
                    "template_id": template_id,
                    "template_name": template_name,
                    "subject": template_subject,
                    "template_text": original_text,
                    "matched_text": cls["text"],
                    "matched_text_norm": pattern,
                    "start_pos": m.start(),
                    "end_pos": m.end(),
                    "context": cls["context"],
                    "subcontext": cls["subcontext"],
                    "risk": cls["risk"],
                    "message_type": cls["message_type"],
                    "text_level": cls["text_level"],
                    "intention": cls["intention"],
                    "action_required": cls["action_required"],
                    "is_urgent": cls["is_urgent"],
                    "has_url": cls["has_url"],
                    "has_credentials": cls["has_credentials"]
                })

df_matches = pd.DataFrame(matches)

# -----------------------------
# 6. Quitar duplicados exactos
# -----------------------------
if not df_matches.empty:
    df_matches = df_matches.drop_duplicates(
        subset=["template_id", "matched_text", "start_pos", "end_pos"]
    ).reset_index(drop=True)

# -----------------------------
# 7. Resumen por template
# -----------------------------
if not df_matches.empty:
    df_summary = (
        df_matches
        .groupby(["template_id", "template_name", "subject"], as_index=False)
        .agg(
            total_matches=("matched_text", "count"),
            contexts_detected=("context", lambda x: ", ".join(sorted(set(x)))),
            max_risk=("risk", lambda x: sorted(set(x))[0] if len(set(x)) == 1 else ", ".join(sorted(set(x))))
        )
    )
else:
    df_summary = pd.DataFrame()

# -----------------------------
# 8. Ver resultados
# -----------------------------
print("Coincidencias encontradas:")
display(df_matches)

print("\nResumen por template:")
display(df_summary)

Coincidencias encontradas:


,template_id,template_name,subject,template_text,matched_text,matched_text_norm,start_pos,end_pos,context,subcontext,risk,message_type,text_level,intention,action_required,is_urgent,has_url,has_credentials
0,64,Demo 1,Seguridad - UTEC,"Estimado Estudiante, Para tu seguridad es nece...",credenciales,credenciales,77,89,Seguridad,Bloqueo,Medio,Neutral,Palabra o frase corta,Referencia,0,0,0,1
1,64,Demo 1,Seguridad - UTEC,"Estimado Estudiante, Para tu seguridad es nece...",contrasena,contrasena,150,160,Seguridad,Alerta,Medio,Neutral,Palabra o frase corta,Referencia,0,0,0,1



Resumen por template:


,template_id,template_name,subject,total_matches,contexts_detected,max_risk
0,64,Demo 1,Seguridad - UTEC,2,Seguridad,Medio


In [9]:
print(df_templates[["id", "name", "text"]].head(20))
print(df_templates["text"].notna().sum())

    id            name                                               text
0    1   Template P1-1  Actualizacion de cuenta corporativa P1-1. Se s...
1    2   Template P1-2  Revision de acceso a VPN P1-2. Se solicita val...
2    3   Template P1-3  Confirmacion de nomina interna P1-3. Se solici...
3    4   Template P1-4  Verificacion de correo empresarial P1-4. Se so...
4    5   Template P1-5  Politica nueva de seguridad P1-5. Se solicita ...
5    6   Template P1-6  Portal de vacaciones y permisos P1-6. Se solic...
6    7   Template P1-7  Actualizacion de beneficios internos P1-7. Se ...
7    8   Template P1-8  Revision de firma electronica P1-8. Se solicit...
8    9   Template P1-9  Confirmacion de acceso remoto P1-9. Se solicit...
9   10  Template P1-10  Validacion de directorio interno P1-10. Se sol...
10  11  Template P1-11  Actualizacion de cuenta corporativa P1-11. Se ...
11  12  Template P1-12  Revision de acceso a VPN P1-12. Se solicita va...
12  13  Template P1-13  Confirmacion d

In [10]:
df_templates["text_norm"] = df_templates["text"].apply(normalize_text)

df_templates["has_text"] = df_templates["text_norm"].str.len() > 0
print(df_templates["has_text"].value_counts())

has_text
True    37
Name: count, dtype: int64


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# unir textos
corpus = list(df_templates["text_norm"]) + list(df_cls["pattern_norm"])

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# separar
tpl_vecs = X[:len(df_templates)]
cls_vecs = X[len(df_templates):]

results = []

for i, tpl_vec in enumerate(tpl_vecs):
    similarities = cosine_similarity(tpl_vec, cls_vecs)[0]

    best_idx = similarities.argmax()
    best_score = similarities[best_idx]

    if best_score > 0.2:  # umbral ajustable
        cls = df_cls.iloc[best_idx]

        results.append({
            "template_id": df_templates.iloc[i]["id"],
            "template_name": df_templates.iloc[i]["name"],
            "predicted_context": cls["context"],
            "risk": cls["risk"],
            "similarity": best_score
        })

df_results = pd.DataFrame(results)

display( df_results )

,template_id,template_name,predicted_context,risk,similarity
0,1,Template P1-1,Bancario,Alto,0.230784
1,2,Template P1-2,Administrativo,Alto,0.205798
2,3,Template P1-3,Administrativo,Medio,0.222693
3,4,Template P1-4,Administrativo,Bajo,0.256289
4,5,Template P1-5,Seguridad,Medio,0.363535
5,8,Template P1-8,Administrativo,Alto,0.289743
6,10,Template P1-10,Administrativo,Bajo,0.391107
7,11,Template P1-11,Bancario,Alto,0.214361
8,13,Template P1-13,Administrativo,Medio,0.208014
9,14,Template P1-14,Administrativo,Bajo,0.237944


In [21]:
import re
import unicodedata
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# =========================================
# 2. CARGAR DATOS
# =========================================
df_train = pd.read_sql("""
    SELECT id, text, context, subcontext, risk, message_type,
           text_level, intention, action_required, is_urgent,
           has_url, has_credentials
    FROM text_clasification
""", engine)

df_templates = pd.read_sql("""
    SELECT id, user_id, name, subject, text, html, modified_date
    FROM templates
""", engine)

# =========================================
# 3. LIMPIEZA DE TEXTO
# =========================================
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()

    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )

    s = re.sub(r'[\r\n\t]+', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s

df_train["text_clean"] = df_train["text"].apply(normalize_text)

# Puedes usar solo text, o combinar subject + text
df_templates["full_text"] = (
    df_templates["subject"].fillna("") + " " +
    df_templates["text"].fillna("")
)

df_templates["text_clean"] = df_templates["full_text"].apply(normalize_text)

# Quitar vacios por seguridad
df_train = df_train[df_train["text_clean"].str.len() > 0].copy()
df_templates = df_templates[df_templates["text_clean"].str.len() > 0].copy()

# =========================================
# 4. DEFINIR X e y
# =========================================
X = df_train["text_clean"]
y = df_train["context"]

# =========================================
# 5. SPLIT TRAIN / TEST
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# 6. PIPELINE DEL MODELO
# =========================================
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),   # palabras y frases cortas
            min_df=2,
            max_df=0.95
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

# =========================================
# 7. ENTRENAR
# =========================================
model.fit(X_train, y_train)

# =========================================
# 8. EVALUAR
# =========================================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

# =========================================
# 9. PREDECIR TEMPLATES
# =========================================
template_preds = model.predict(df_templates["text_clean"])
template_probs = model.predict_proba(df_templates["text_clean"])

class_names = model.named_steps["clf"].classes_

df_pred = df_templates[["id", "name", "subject", "text"]].copy()
df_pred["predicted_context"] = template_preds
df_pred["max_probability"] = template_probs.max(axis=1)

# Agregar probabilidad por clase
prob_df = pd.DataFrame(
    template_probs,
    columns=[f"prob_{c}" for c in class_names]
)

df_pred = pd.concat(
    [df_pred.reset_index(drop=True), prob_df.reset_index(drop=True)],
    axis=1
)

# =========================================
# 10. CLASIFICACION FINAL DE CONFIANZA
# =========================================
def confidence_label(p):
    if p >= 0.80:
        return "Alta"
    elif p >= 0.55:
        return "Media"
    else:
        return "Baja"

df_pred["confidence_level"] = df_pred["max_probability"].apply(confidence_label)

# =========================================
# 11. VER RESULTADOS
# =========================================
print("\nPredicciones sobre templates:")
display(df_pred[[
    "id", "name", "predicted_context", "max_probability", "confidence_level"
]].head(20))

# =========================================
# 12. GUARDAR RESULTADOS EN MYSQL
# =========================================
""" df_pred.to_sql(
    "template_predictions_context",
    con=engine,
    if_exists="replace",
    index=False
)
 """


Accuracy: 1.0

Classification report:
                precision    recall  f1-score   support

     Academico       1.00      1.00      1.00       532
Administrativo       1.00      1.00      1.00       420
      Bancario       1.00      1.00      1.00       432
    Financiero       1.00      1.00      1.00       432
     Seguridad       1.00      1.00      1.00       432
       Soporte       1.00      1.00      1.00       420

      accuracy                           1.00      2668
     macro avg       1.00      1.00      1.00      2668
  weighted avg       1.00      1.00      1.00      2668


Predicciones sobre templates:


,id,name,predicted_context,max_probability,confidence_level
0,1,Template P1-1,Bancario,0.688068,Media
1,2,Template P1-2,Administrativo,0.432966,Baja
2,3,Template P1-3,Administrativo,0.853307,Alta
3,4,Template P1-4,Seguridad,0.860623,Alta
4,5,Template P1-5,Seguridad,0.490578,Baja
5,6,Template P1-6,Financiero,0.736321,Media
6,7,Template P1-7,Administrativo,0.963702,Alta
7,8,Template P1-8,Administrativo,0.957902,Alta
8,9,Template P1-9,Soporte,0.536785,Baja
9,10,Template P1-10,Administrativo,0.750307,Media


' df_pred.to_sql(\n    "template_predictions_context",\n    con=engine,\n    if_exists="replace",\n    index=False\n)\n '